<a href="https://colab.research.google.com/github/leticialindona/prova_pestana/blob/main/notebook_consulta.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook de consulta

Templates genéricos por técnica. Copie a célula, troque o que está marcado
com 🔧 e rode. Explicação completa nos arquivos `.md` da mesma pasta.

**Índice:** 1 imports · 2 diagnóstico · 3 limpeza · 4 datas mistas ·
5 coluna calculada · 6 agrupar · 7 gráficos · 8 datas/recortes ·
9 features sem vazamento · 10 regressão · 11 classificação ·
12 matriz+cortes · 13 importâncias · 14 clusterização

1. Imports

In [ ]:
import pandas as pd

import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import make_pipeline

from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier

from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier

from sklearn.naive_bayes import GaussianNB

from sklearn.cluster import KMeans

from sklearn.metrics import (
    mean_absolute_error, r2_score,
    precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, silhouette_score,
)


pd.set_option("display.max_columns", None)

FONTE = "Fonte: dados sintéticos do Festival ViraBairro (2026)"   # 🔧
def fonte():
    plt.figtext(0.5, -0.05, FONTE, ha="center", fontsize=9, style="italic")

2. Diagnóstico da base

In [ ]:
df = pd.read_csv("dados/ARQUIVO.csv")          # 🔧

print(df.head())
print(f"Linhas: {df.shape[0]}, Colunas: {df.shape[1]}")

ausencias = df.isna().sum()
print(ausencias[ausencias > 0])

# reconhecimento rápido
print(df.dtypes)
print(df["CATEGORIA"].unique())                # 🔧 ver grafias inconsistentes
print(df.duplicated(subset="ID").sum())        # 🔧

3. Limpeza

In [ ]:
df = df.drop_duplicates(subset="ID").copy()                        # 🔧

# padronizar texto
df["CATEGORIA"] = df["CATEGORIA"].str.strip().str.lower().str.title()
print(sorted(df["CATEGORIA"].unique()))        # confira

# texto -> número
for col in ["NUM1", "NUM2", "NUM3"]:           # 🔧
    df[col] = pd.to_numeric(df[col], errors="coerce")

# ausências e inválidos
df = df.dropna(subset=["COL_ESSENCIAL"])       # 🔧 descartar
df["COL"] = df["COL"].fillna(df["COL"].median())   # 🔧 preencher
df = df[df["DENOMINADOR"] > 0].copy()          # 🔧 inválidos

4. Datas em formatos misturados

In [ ]:
iso = pd.to_datetime(df["DATA"], errors="coerce", format="%Y-%m-%d")   # 🔧
br  = pd.to_datetime(df["DATA"], errors="coerce", format="%d/%m/%Y")
df["DATA"] = iso.fillna(br)

# CONFIRA sempre
print("não convertidas:", df["DATA"].isna().sum())
print(df["DATA"].min(), "→", df["DATA"].max())

5. coluna calculada

In [ ]:
df["taxa"] = (df["A"] + df["B"]) / df["C"] * 100      # 🔧 fórmula do enunciado

print(df["taxa"].describe())
print("infinitos:", np.isinf(df["taxa"]).sum())

6. Agrupar e montar tabelas

In [ ]:
# uma categoria
resumo = (df.groupby("CATEGORIA")["taxa"]
          .agg(publicacoes="count", mediana="median")
          .sort_values("mediana", ascending=False))
print(resumo)

# duas categorias
resumo2 = (df.groupby(["CAT_A", "CAT_B"])["taxa"]
           .agg(publicacoes="count", mediana="median")
           .sort_values("mediana", ascending=False)
           .reset_index())
print(resumo2)

# em matriz
print(df.pivot_table(index="CAT_A", columns="CAT_B", values="taxa", aggfunc="median").round(2))

# colunas diferentes, funções diferentes
print(df.groupby("CHAVE").agg(
    qtd=("ID", "count"), media=("taxa", "mean"), total=("alcance", "sum")
).sort_index())

7. Gráficos

In [ ]:
# BARRAS
plt.figure(figsize=(8, 5))
plt.bar(resumo.index, resumo["mediana"], color="#4C72B0")
plt.title("TÍTULO QUE COMUNICA A PERGUNTA")    # 🔧
plt.xlabel("EIXO X"); plt.ylabel("EIXO Y")     # 🔧
fonte(); plt.tight_layout(); plt.show()

In [ ]:
# BARRAS COMBINADAS (duas categorias)
resumo2 = resumo2.copy()
resumo2["rotulo"] = resumo2["CAT_A"] + " – " + resumo2["CAT_B"]

plt.figure(figsize=(11, 6))
plt.bar(resumo2["rotulo"], resumo2["mediana"], color="#55A868")
plt.title("TÍTULO"); plt.xlabel("Combinação"); plt.ylabel("Mediana (%)")
plt.xticks(rotation=45, ha="right")
fonte(); plt.tight_layout(); plt.show()

In [ ]:
# LINHA
plt.figure(figsize=(9, 5))
plt.plot(diario.index, diario["media"], marker="o", color="#4C72B0")
plt.title("TÍTULO"); plt.xlabel("Dia"); plt.ylabel("EIXO Y")
plt.xticks(rotation=45, ha="right")
fonte(); plt.tight_layout(); plt.show()

In [ ]:
# BARRAS HORIZONTAIS (importâncias) — ordenar CRESCENTE
top = importancias.head(5).sort_values()
plt.figure(figsize=(8, 5))
plt.barh(top.index, top.values, color="#C44E52")
plt.title("Características mais importantes")
plt.xlabel("Importância"); plt.ylabel("Característica")
plt.tight_layout(); plt.show()

In [ ]:
# DISPERSÃO real x previsto
plt.figure(figsize=(6, 6))
plt.scatter(y_teste, y_previsto, alpha=0.6, color="#4C72B0")
lo = min(y_teste.min(), y_previsto.min()); hi = max(y_teste.max(), y_previsto.max())
plt.plot([lo, hi], [lo, hi], "--", color="gray", label="previsão = valor real")
plt.title("Valores reais vs. previstos")
plt.xlabel("Valor real"); plt.ylabel("Valor previsto")
plt.legend(); plt.tight_layout(); plt.show()

8. Datas e recortes

In [ ]:
df["dia"] = df["DATA"].dt.date

diario = (df.groupby("dia")
          .agg(publicacoes=("ID", "count"),
               media=("taxa", "mean"),
               alcance_total=("alcance", "sum"))
          .sort_index())
print(diario)

# recorte com condição (cada condição entre parênteses!)
rec = df[(df["formato"] == "reel") & (df["hora"] >= 18)]      # 🔧
print(len(rec), "linhas no recorte")

top5 = rec.nlargest(5, "taxa")[["ID", "dia", "hora", "CATEGORIA", "taxa"]]   # 🔧
print(top5)

9. Features sem vazamento

In [ ]:
FEATURES = ["tema", "formato", "seguidores_autor", "videos_autor",
            "tamanho_legenda", "n_emojis", "n_hashtags", "hora",
            "dia_semana", "duracao_segundos"]      # 🔧 as permitidas pelo enunciado

# NUNCA: alcance, interacoes, compartilhamentos, salvamentos, o alvo, o id
X = pd.get_dummies(df[FEATURES], columns=["tema", "formato"],
                   drop_first=True, dtype=int)
print(X.shape, "->", X.columns.tolist()[:8], "...")

10. Regressão ( prever número )

In [ ]:
y = df["ALVO_NUMERICO"]                                   # 🔧

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42)                # sem stratify em regressão

linear = LinearRegression().fit(X_treino, y_treino)
arvore = DecisionTreeRegressor(max_depth=4, random_state=42).fit(X_treino, y_treino)

tabela = pd.DataFrame([
    {"modelo": nome,
     "MAE": mean_absolute_error(y_teste, m.predict(X_teste)),
     "R2":  r2_score(y_teste, m.predict(X_teste))}
    for nome, m in [("Regressão linear", linear), ("Árvore de regressão", arvore)]
]).sort_values("MAE")
print(tabela)

melhor = linear if tabela.iloc[0]["modelo"] == "Regressão linear" else arvore
y_previsto = melhor.predict(X_teste)
print("MODELO ESCOLHIDO:", tabela.iloc[0]["modelo"])

11. Classificação ( prever categoria )

In [ ]:
limite = df["taxa_engajamento_pct"].quantile(0.75)        # 🔧
df["ALVO"] = (df["taxa_engajamento_pct"] > limite).astype(int)
y = df["ALVO"]
print("limite:", round(limite, 2)); print(y.value_counts())

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)     # stratify!

modelos = {
    "Regressão logística": make_pipeline(          # pipeline = escalona + ajusta
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced")),
    "Árvore de classificação": DecisionTreeClassifier(
        max_depth=4, random_state=42, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        random_state=42, class_weight="balanced"),
    # "Extra Trees": ExtraTreesClassifier(random_state=42, class_weight="balanced"),
    # "AdaBoost": AdaBoostClassifier(random_state=42),     # SEM class_weight
    # "Gaussian Naive Bayes": GaussianNB(),                # SEM class_weight
}
for m in modelos.values():
    m.fit(X_treino, y_treino)

tabela = pd.DataFrame([
    {"modelo": nome,
     "precisao": precision_score(y_teste, m.predict(X_teste), zero_division=0),
     "recall":   recall_score(y_teste, m.predict(X_teste), zero_division=0),
     "f1":       f1_score(y_teste, m.predict(X_teste), zero_division=0)}
    for nome, m in modelos.items()
]).sort_values("f1", ascending=False)
print(tabela)

12. Matriz de confusão e cortes

In [ ]:
melhor_nome = tabela.iloc[0]["modelo"]
melhor = modelos[melhor_nome]

cm = confusion_matrix(y_teste, melhor.predict(X_teste))
ConfusionMatrixDisplay(cm, display_labels=["Não", "Sim"]).plot(cmap="Blues")
plt.title(f"Matriz de confusão — {melhor_nome}")
plt.show()
print(cm)   # [[VN, FP], [FN, VP]]

# cortes na logística já ajustada
prob = modelos["Regressão logística"].predict_proba(X_teste)[:, 1]
tabela_cortes = pd.DataFrame([
    {"corte": corte,
     "precisao": precision_score(y_teste, (prob >= corte).astype(int), zero_division=0),
     "recall":   recall_score(y_teste, (prob >= corte).astype(int), zero_division=0),
     "f1":       f1_score(y_teste, (prob >= corte).astype(int), zero_division=0)}
    for corte in [0.50, 0.30]                              # 🔧
])
print(tabela_cortes)

13. Importância das características

In [ ]:
# árvore / floresta
arvore_clf = modelos["Árvore de classificação"]
importancias = pd.Series(arvore_clf.feature_importances_,
                         index=X_treino.columns).sort_values(ascending=False)
print(importancias.head(5))

# regressão logística dentro de pipeline
# log = modelos["Regressão logística"].named_steps["logisticregression"]
# importancias = pd.Series(log.coef_[0], index=X_treino.columns).abs().sort_values(ascending=False)

14. Clusterização ( sem alvo )

In [ ]:
COLS = ["seguidores_autor", "tamanho_legenda", "n_hashtags",
        "taxa_engajamento_pct"]                            # 🔧

base = df[COLS].dropna().copy()
Xs = StandardScaler().fit_transform(base)                  # escalonar é obrigatório

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(Xs)
    print(f"k={k} | inércia={km.inertia_:.1f} | silhueta={silhouette_score(Xs, km.labels_):.3f}")

km = KMeans(n_clusters=3, random_state=42, n_init=10).fit(Xs)   # 🔧 k escolhido
base["cluster"] = km.labels_
print(base.groupby("cluster")[COLS].mean().round(2))
print(base["cluster"].value_counts().sort_index())